# import

In [1]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score
)
import argparse

import plotly.express as px

# funciones extra

In [2]:
def graficar_3d(X, color):
    fig = px.scatter_3d(
        X,
        x='wind_speed_ms',
        y='wave_period_s',
        z='wave_height_m',
        color=color,
        opacity=0.5
    )
    fig.update_traces(marker_size=3)
    fig.update_layout(
        scene=dict(
            aspectmode='cube'
        )
    )

    return fig

In [3]:
def graficar_2d(X, color):
    fig = px.scatter(
        X,
        x='datetime',
        y='wave_height_m',
        color=color,
        opacity=0.5
    )
    fig.update_traces(marker_size=3)

    return fig

# configuraciones

In [ ]:
ambiente = 'project'

EXTREME_THRESHOLD = 0.95
EVERY_N_YEARS = 10
PORCENTAJE_SAMPLE_DATA_MONTH = 1
RANDOM_SEED = 0
TEST_SIZE = 0.2

N_CLUSTERS = 6
EXTREME_FEATURES = ['wave_energy', 'wave_power_kW_m']
MODEL_FEATURES = [
    'wind_speed_ms', 
    'wave_energy', 'wave_height_m', 'wave_period_s', 'wave_power_kW_m'
]
SCALED_MODEL_FEATURES = [f'{feature}_scaled' for feature in MODEL_FEATURES]

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_classifier/wave_classifier{{}}.pkl'
else:
    base_path = Path.cwd().parent
    model_path = f'{base_path}/wave_classifier/wave_classifier{{}}.pkl'


# proceso separado por celdas

In [ ]:
X_train, X_test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_SEED)

In [ ]:
threshold_energia = X_train["wave_energy"].quantile(EXTREME_THRESHOLD)
threshold_potencia = X_train["wave_power_kW_m"].quantile(EXTREME_THRESHOLD)

In [ ]:
def es_extremo(X):
    return (
        (X["wave_energy"] >= threshold_energia) |
        (X["wave_power_kW_m"] >= threshold_potencia)
    )

In [ ]:
scaler = StandardScaler()
scaler.fit(
    X_train.loc[~es_extremo(X_train), MODEL_FEATURES]
)

In [ ]:
def procesar(X):
    x = X.copy()
    x["es_extremo"] = es_extremo(x)
    x_extremo = x[x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])
    x_no_extremo = x[~x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])

    temp_scaled = scaler.transform(x_no_extremo[MODEL_FEATURES])
    df_temp_scaled = pd.DataFrame(temp_scaled, columns=SCALED_MODEL_FEATURES)

    x_no_extremo = pd.concat([x_no_extremo.reset_index(drop=True), df_temp_scaled], axis=1)
    
    return x_no_extremo, x_extremo

In [ ]:
X_train_no_extremo, X_train_extremo = procesar(X_train)
X_test_no_extremo, X_test_extremo = procesar(X_test)

In [ ]:
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="full",
    random_state=RANDOM_SEED
)

In [ ]:
gmm.fit(X_train_no_extremo[SCALED_MODEL_FEATURES])

In [ ]:
X_train_no_extremo["gmm_cluster"] = gmm.predict(X_train_no_extremo[SCALED_MODEL_FEATURES])
X_train_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_train_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

In [ ]:
cluster_summary = (
    X_train_no_extremo.groupby("gmm_cluster")[EXTREME_FEATURES]
    .mean()
    .sort_values(EXTREME_FEATURES)
)
cluster_order = {
    old_cluster: f'{new_cluster + 1}'
    for new_cluster, old_cluster in enumerate(cluster_summary.index)
}

In [ ]:
X_test_no_extremo["gmm_cluster"] = gmm.predict(X_test_no_extremo[SCALED_MODEL_FEATURES])
X_test_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_test_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

In [ ]:
q_5 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)
q_10 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)
q_15 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)

In [ ]:
if not (
    all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)>=0.7) and
    all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)>=0.8) and
    all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)>=0.9) 
):
    print(f'El modelo GMM no es lo suficientemente bueno, saltando...')
else:
        print(f'Modelo GMM cumple con los criterios')

print(f'Quantiles: 0.05: {q_5}, 0.10: {q_10}, 0.15: {q_15}')

In [ ]:
X_test_no_extremo = X_test_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
X_test_extremo = X_test_extremo.assign(gmm_cluster='7')

X_train_no_extremo = X_train_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
X_train_extremo = X_train_extremo.assign(gmm_cluster='7')

In [ ]:
X_train_classified = pd.concat(
    [
        X_train_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
        X_train_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
    ],
    ignore_index=True
)

X_test_classified = pd.concat(
    [
        X_test_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
        X_test_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
    ],
    ignore_index=True
)

# proceso en 1 funcion

In [93]:
def proceso(df):
    def es_extremo(X):
        return (
            (X["wave_energy"] >= threshold_energia) &
            (X["wave_power_kW_m"] >= threshold_potencia)
        )
    def procesar(X):
        x = X.copy()
        x["es_extremo"] = es_extremo(x)
        x_extremo = x[x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])
        x_no_extremo = x[~x['es_extremo']].reset_index(drop=True).drop(columns=['es_extremo'])

        temp_scaled = scaler.transform(x_no_extremo[MODEL_FEATURES])
        df_temp_scaled = pd.DataFrame(temp_scaled, columns=SCALED_MODEL_FEATURES)

        x_no_extremo = pd.concat([x_no_extremo.reset_index(drop=True), df_temp_scaled], axis=1)
        
        return x_no_extremo, x_extremo
    
    X_train, X_test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_SEED)

    threshold_energia = X_train["wave_energy"].quantile(EXTREME_THRESHOLD)
    threshold_potencia = X_train["wave_power_kW_m"].quantile(EXTREME_THRESHOLD)

    scaler = StandardScaler()
    scaler.fit(
        X_train.loc[~es_extremo(X_train), MODEL_FEATURES]
    )

    X_train_no_extremo, X_train_extremo = procesar(X_train)
    X_test_no_extremo, X_test_extremo = procesar(X_test)

    gmm = GaussianMixture(
        n_components=N_CLUSTERS,
        covariance_type="full",
        random_state=RANDOM_SEED
    )

    gmm.fit(X_train_no_extremo[SCALED_MODEL_FEATURES])

    X_train_no_extremo["gmm_cluster"] = gmm.predict(X_train_no_extremo[SCALED_MODEL_FEATURES])
    X_train_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_train_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

    cluster_summary = (
        X_train_no_extremo.groupby("gmm_cluster")[EXTREME_FEATURES]
        .mean()
        .sort_values(EXTREME_FEATURES)
    )
    cluster_order = {
        old_cluster: f'{new_cluster + 1}'
        for new_cluster, old_cluster in enumerate(cluster_summary.index)
    }

    X_test_no_extremo["gmm_cluster"] = gmm.predict(X_test_no_extremo[SCALED_MODEL_FEATURES])
    X_test_no_extremo["gmm_cluster_probability"] = gmm.predict_proba(X_test_no_extremo[SCALED_MODEL_FEATURES]).max(axis=1)

    q_5 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)
    q_10 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)
    q_15 = X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)
    if not (
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.05)>=0.7) and
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.10)>=0.8) and
        all(X_test_no_extremo.groupby('gmm_cluster')['gmm_cluster_probability'].quantile(0.15)>=0.9) 
    ):
        print(f'El modelo GMM no es lo suficientemente bueno, saltando...')
    else:
            print(f'Modelo GMM cumple con los criterios')

    print(f'Quantiles: 0.05: {q_5}, 0.10: {q_10}, 0.15: {q_15}')

    X_test_no_extremo = X_test_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
    X_test_extremo = X_test_extremo.assign(gmm_cluster='7')

    X_train_no_extremo = X_train_no_extremo.assign(gmm_cluster=lambda df: df["gmm_cluster"].map(cluster_order))
    X_train_extremo = X_train_extremo.assign(gmm_cluster='7')

    X_train_classified = pd.concat(
        [
            X_train_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
            X_train_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
        ],
        ignore_index=True
    )

    X_test_classified = pd.concat(
        [
            X_test_no_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']],
            X_test_extremo[['coast_name', 'datetime'] + MODEL_FEATURES + ['gmm_cluster']]
        ],
        ignore_index=True
    )

    return X_train_classified, X_test_classified

# obtener datos

In [39]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(set(MODEL_FEATURES + EXTREME_FEATURES))},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [40]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA_MONTH for row in data.select('coast_year_month').distinct().collect()}

df = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
).toPandas()

# prueba todos los datos

In [94]:
X_train_classified, X_test_classified = proceso(df)

Modelo GMM cumple con los criterios
Quantiles: 0.05: gmm_cluster
0    0.866726
1    0.975455
2    0.821637
3    0.860435
4    0.972087
5    0.853496
Name: gmm_cluster_probability, dtype: float64, 0.10: gmm_cluster
0    0.970710
1    0.999622
2    0.962782
3    0.973461
4    0.999903
5    0.969485
Name: gmm_cluster_probability, dtype: float64, 0.15: gmm_cluster
0    0.995446
1    0.999997
2    0.992699
3    0.996314
4    1.000000
5    0.993861
Name: gmm_cluster_probability, dtype: float64


In [95]:
matamoros = X_train_classified[X_train_classified['coast_name']=='Matamoros']

In [96]:
#agrupar por mes y ver la distribución de clusters
cluster_month = matamoros.groupby([matamoros['datetime'].dt.month, 'gmm_cluster']).size().unstack(fill_value=0)

In [97]:
px.line(cluster_month).show()

# proceso solo matamoros

In [98]:
df_2 = df[df['coast_name']=='Matamoros'].reset_index(drop=True)

In [99]:
X_train_classified, X_test_classified = proceso(df_2)

Modelo GMM cumple con los criterios
Quantiles: 0.05: gmm_cluster
0    0.981507
1    0.838449
2    0.841934
3    0.977828
4    0.851417
5    0.795430
Name: gmm_cluster_probability, dtype: float64, 0.10: gmm_cluster
0    0.999924
1    0.964845
2    0.970578
3    0.999669
4    0.967950
5    0.948817
Name: gmm_cluster_probability, dtype: float64, 0.15: gmm_cluster
0    1.000000
1    0.993339
2    0.995261
3    0.999998
4    0.995773
5    0.987521
Name: gmm_cluster_probability, dtype: float64


In [100]:
matamoros = X_train_classified[X_train_classified['coast_name']=='Matamoros']

In [101]:
cluster_month = matamoros.groupby([matamoros['datetime'].dt.month, 'gmm_cluster']).size().unstack(fill_value=0)

In [102]:
px.line(cluster_month).show()